# sample_container_sequence

Generated from `alteryx-workflows\sample_container_sequence.yxmd` by compiler 2.2.0.

## How to use this notebook

1. Review and complete the source configuration.
2. Run cells from top to bottom.
3. Inspect the output previews and validate semantic parity.

> **Important:** Generated transformations are unvalidated. Confirm schemas, row counts, business rules, and destinations before promotion.


## Runtime imports

Imports and runtime helpers required by the generated PySpark transformations.


In [ ]:
from pyspark.sql import Window, functions as F


## Source configuration

Replace each empty value with an accessible Fabric or OneLake path before running the ingestion cells.

| Tool ID | Generated DataFrame |
| --- | --- |
| `1` | `orders` |
| `5` | `customers` |


In [ ]:
source_paths = {
    "1": "",  # orders
    "5": "",  # customers
}


## Source ingestion

Loads and validates source datasets before transformation.


### Source ingestion 1 of 1

Produces `orders_Path`, `orders`, `customers_Path`, `customers`.


In [ ]:
orders_Path = source_paths.get('1')
if not orders_Path: raise ValueError("Configure source_paths['1'] before running")
orders = spark.read.option('header', True).option('inferSchema', True).csv(orders_Path)
customers_Path = source_paths.get('5')
if not customers_Path: raise ValueError("Configure source_paths['5'] before running")
customers = spark.read.option('header', True).option('inferSchema', True).csv(customers_Path)


## Transformations

Applies the translated Alteryx operations in dependency order.


### Transformations 1 of 1

Produces `keep_order_fields`, `amount_and_units_1`, `amount_and_units_2`, `positive_amounts`, and 4 more.


In [ ]:
keep_order_fields = orders.select(*["OrderID", "CustomerID", "SKU", "Amount", "Units"])
amount_and_units_1 = keep_order_fields.withColumn("AmountNum", F.col("Amount").cast('double'))
amount_and_units_2 = amount_and_units_1.withColumn("UnitsNum", F.col("Units").cast('double'))
positive_amounts = amount_and_units_2.filter(F.col("AmountNum") > 0)
keep_customer_fields = customers.select(*["CustomerID", "CustomerName", "Tier", "Status"])
active_customers = keep_customer_fields.filter(F.col("Status") == "Active")
match_on_customer = positive_amounts.join(active_customers, (positive_amounts["CustomerID"] == active_customers["CustomerID"]), 'inner').drop(active_customers["CustomerID"])
line_value = match_on_customer.withColumn("LineValue", F.col("AmountNum") * F.col("UnitsNum"))


## Output preview

Displays terminal DataFrames for review. Replace preview-only output with approved destination writes after validation.


### Output preview 1 of 1


In [ ]:
display(line_value)
